In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail \
     --retry 999 \
     --retry-all-errors \
     --retry-delay 5 \
     --retry-max-time 600 \
     http://gateway:8001/api/games


In [ ]:
import os
os.environ['AGI_ARC_REPO'] = '/kaggle/input/datasets/suminshim/agi-arc-submit/agi-arc-main'
os.environ.setdefault('ARC_VLM_PRINT_OUTPUT', '0')
os.environ.setdefault('ARC_VLM_ACTIONS_PER_CALL', '3')
os.environ.setdefault('ARC_VLM_MAX_NEW_TOKENS', '512')
os.environ.setdefault('ALLOW_MODEL_DOWNLOAD', '0')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('HF_DATASETS_OFFLINE', '1')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_agi_arc_repo() -> Path:
    candidates = []
    env_repo = os.getenv('AGI_ARC_REPO')
    if env_repo:
        candidates.append(Path(env_repo))
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(sorted(path for path in input_root.iterdir() if path.is_dir()))
        candidates.extend(sorted(input_root.rglob('*')))
    seen = set()
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (candidate / 'vlm-agi').exists() and (candidate / 'scripts' / 'kaggle_vlm_eval_agent.py').exists():
            return candidate
        if (candidate / 'src' / 'arc_agi3').exists() and (candidate / 'scripts' / 'kaggle_competition_submission.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the agi-arc repo with VLM submission files in /kaggle/input.')


def find_vlm_model_path() -> Path:
    explicit = os.getenv('LOCAL_VLM_MODEL_PATH')
    if explicit and (Path(explicit) / 'config.json').exists():
        return Path(explicit)

    input_root = Path('/kaggle/input')
    if not input_root.exists():
        raise FileNotFoundError('No /kaggle/input directory found for model discovery.')

    for config_path in input_root.rglob('config.json'):
        parent = config_path.parent
        parent_text = str(parent).lower()
        if 'qwen' in parent_text and 'vl' in parent_text:
            return parent

    raise FileNotFoundError('Could not locate a Qwen VL model directory under /kaggle/input.')


if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    COMPETITION_ROOT = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    REPO_SOURCE = find_agi_arc_repo()
    MODEL_PATH = find_vlm_model_path()
    WORKING_COPY = '/kaggle/working/ARC-AGI-3-Agents-agi-arc-vlm-submission'

    os.environ['AGI_ARC_REPO'] = str(REPO_SOURCE)
    os.environ['LOCAL_VLM_MODEL_PATH'] = str(MODEL_PATH)
    os.environ.setdefault('ARC_API_KEY', 'test-key-123')

    print(f'Using agi-arc source from input: {REPO_SOURCE}')
    print(f'Using VLM model from input: {MODEL_PATH}')
    print(f'Writable execution copy will be created at: {WORKING_COPY}')

    subprocess.run(
        [
            sys.executable,
            str(REPO_SOURCE / 'scripts' / 'kaggle_competition_submission.py'),
            '--repo-root',
            str(REPO_SOURCE),
            '--competition-root',
            COMPETITION_ROOT,
            '--working-root',
            '/kaggle/working',
            '--description',
            'ARC-AGI-3-Agents-agi-arc-vlm-submission',
            '--agent-script',
            'scripts/kaggle_vlm_eval_agent.py',
        ],
        check=True,
    )


In [ ]:
# Non-rerun mode: produce a dummy submission so the notebook
# has valid output for the initial commit / save.
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame({'task_id': ['dummy'], 'output': ['[[0]]']}).to_parquet(
        'submission.parquet', index=False)
    print('Non-competition mode: wrote dummy submission.parquet')
